# N-gram Tracing

This notebook will be used to test out n-gram tracing for use with author verification methods. The end goal is to ensure the code works to find common n-grams between two texts and that we can return the text prior to those n-grams.

In [77]:
import sys
import os

import pandas as pd

from from_root import from_root

sys.path.insert(0, str(from_root("src")))

from model_loading import load_model
from read_and_write_docs import read_jsonl, read_txt, read_rds
from n_gram_tracing import (
    common_ngrams,
    filter_len_common_ngrams,
    tokenize_to_tokens,
    tokens_to_ids,
    tokens_to_text,
    find_all_token_ngram_spans,
    texts_before_each_token_ngram,
    texts_around_each_token_ngram,
    get_trimmed_context_before_span
)
from utils import build_metadata_df, apply_temp_doc_id

### Load data

In [78]:
base_loc = "/Volumes/BCross"

if not os.path.exists(base_loc):
    print("Using Offline Location")
    base_loc = "/Users/user/Documents/uni_work_offline"
else:
    print("Using Volume Location")
    
corpus = "Wiki"
data_type = "test"
model_name = "gpt2"

tokenizer, model = load_model(f"{base_loc}/models/{model_name}")
print(f"{model_name} Loaded")

known_df = read_jsonl(f"{base_loc}/datasets/author_verification/{data_type}/{corpus}/known_raw.jsonl")
unknown_df = read_jsonl(f"{base_loc}/datasets/author_verification/{data_type}/{corpus}/unknown_raw.jsonl")
metadata = read_rds(f"{base_loc}/datasets/author_verification/{data_type}/metadata.rds")

known_df = apply_temp_doc_id(known_df)
unknown_df = apply_temp_doc_id(unknown_df)

print(f"{data_type} | {corpus} | Data Loaded")

problems_str = read_txt(f"{base_loc}/datasets/author_verification/{data_type}/{corpus}/agg_problem_list.txt")
problems = problems_str.split("\n")
print(f"Number of Problems: {len(problems)}")

Using Offline Location
gpt2 Loaded
test | Wiki | Data Loaded
Number of Problems: 225


### Filter for test problem

In [112]:
selected_problem

'Hodja_Nasreddin vs Hodja_Nasreddin'

In [86]:
selected_problem = problems[0]

filtered_metadata = metadata[
    (metadata['corpus'] == corpus)
    & (metadata['problem'] == selected_problem)
].copy()
filtered_metadata = build_metadata_df(filtered_metadata, known_df, unknown_df)

known_author = filtered_metadata['known_author'].iloc[0]
unknown_author = filtered_metadata['unknown_author'].iloc[0]

selected_known = known_df[known_df['author'] == known_author]
selected_unknown = unknown_df[unknown_df['author'] == unknown_author]

unknown_doc = selected_unknown['doc_id'].iloc[0]
unknown_text = selected_unknown['text'].iloc[0]

num_known_problems = len(selected_known)
print(f"There are {num_known_problems} known texts in the problem")

There are 3 known texts in the problem


## Get Common N-Grams

Here we get the n-grams in common between the two texts.

In [ ]:
ngram_list = []
problem_metadata_list = []

for i in range(1, num_known_problems + 1):
    print(f"Working on doc {i}")
    known_doc = selected_known['doc_id'].iloc[i-1]
    known_text = selected_known['text'].iloc[i - 1]
    
    # Perform a try/except to try to find common n-grams
    # Leave a flag if unable to find them
    try:
        common = common_ngrams(
            text1=known_text,
            text2=unknown_text,
            tokenizer=tokenizer,
            include_subgrams=False,
            lowercase=True
        )
        ngrams_found = True
    except:
        common = []
        ngrams_found = False
    ngrams_shared = len(common)
    ngram_list.extend(common)
    
    row = {
        "data_type": data_type,
        "corpus": corpus,
        "problem": selected_problem,
        "known_doc": known_doc,
        "unknown_doc": unknown_doc,
        "known_author": known_author,
        "unknown_author": unknown_author,
        "target": known_author == unknown_author,
        "ngrams_found": ngrams_found,
        "num_ngrams": ngrams_shared,
    }
    problem_metadata_list.append(row)
    
problem_metadata = pd.DataFrame(problem_metadata_list)
# Only keep the distinct list
distinct_ngram_list = [list(x) for x in dict.fromkeys(tuple(x) for x in ngram_list)]

# Sort first by the number of tokens and second by character length
distinct_ngram_list = sorted(
    distinct_ngram_list,
    key=lambda x: (len(x), sum(len(str(token)) for token in x))
)

# Filter by token length
filtered_ngrams = filter_len_common_ngrams(
    distinct_ngram_list,
    min_len=None,
    max_len=None
)

Working on doc 1
Working on doc 2
Working on doc 3


In [93]:
problem_metadata

,data_type,corpus,problem,known_doc,unknown_doc,known_author,unknown_author,target,ngrams_found,num_ngrams
0,test,Wiki,Hodja_Nasreddin vs Hodja_Nasreddin,hodja_nasreddin_text_1,hodja_nasreddin_text_3,Hodja_Nasreddin,Hodja_Nasreddin,True,True,59
1,test,Wiki,Hodja_Nasreddin vs Hodja_Nasreddin,hodja_nasreddin_text_10,hodja_nasreddin_text_3,Hodja_Nasreddin,Hodja_Nasreddin,True,True,58
2,test,Wiki,Hodja_Nasreddin vs Hodja_Nasreddin,hodja_nasreddin_text_11,hodja_nasreddin_text_3,Hodja_Nasreddin,Hodja_Nasreddin,True,True,51


In [110]:
overall_problem_metadata = (
    problem_metadata[['data_type', 'corpus', 'problem', 'known_author', 'unknown_author', 'target']]
    .drop_duplicates()
)
overall_problem_metadata['num_problems'] = len(problem_metadata)
overall_problem_metadata['ngrams_found'] = sum(problem_metadata['ngrams_found'])
overall_problem_metadata['completed'] = overall_problem_metadata['num_problems'] == overall_problem_metadata['ngrams_found']

In [111]:
overall_problem_metadata

,data_type,corpus,problem,known_author,unknown_author,target,num_problems,ngrams_found,completed
0,test,Wiki,Hodja_Nasreddin vs Hodja_Nasreddin,Hodja_Nasreddin,Hodja_Nasreddin,True,3,3,True


## Include n-gram in Result

Now we test including the n-gram in the result, either way works since probably apend back on.

In [58]:
filtered_ngrams[1:10]

[['!'], ['d'], ['if'], ["Ġ'"], ['Ġa'], ['Ġi'], ['Ġe'], ['as'], ['Ġan']]

In [69]:
unknown_tokens = tokenize_to_tokens(unknown_text, tokenizer=tokenizer)

In [70]:
find_all_token_ngram_spans(unknown_tokens, ['Ġa'])

[(151, 152),
 (161, 162),
 (240, 241),
 (252, 253),
 (297, 298),
 (317, 318),
 (467, 468),
 (485, 486),
 (492, 493)]

In [68]:
example_texts = texts_around_each_token_ngram(unknown_text, ['Ġas'], tokenizer=tokenizer)

for i in range(0, len(example_texts)):
    print(f"Text {i} - {example_texts[i][-20:]}")

Text 0 - e very different, as
Text 1 - ooks that qualify as
Text 2 - te links to, just as
